# F2: Grad-CAM, per-phase attribution and example images

**Revision plan, 00 log section 25 (approved 19 Oct 2026).** Completes Reviewer 2 point 6 ("Grad-CAM visualisations are
qualitative only; attention maps cannot represent cross-phase correspondence") and Reviewer 4 point 10 (the old Figure 15
showed internal patients labelled as external; low-quality Figures 3 and 13–15).

- **Model:** the main phase-dropout CNN (E3 arm D), frozen; nothing is trained or selected. Held-out patients (56) and
  MCT-LTDiag (203) with the 5-model ensemble. Gates: the ensemble must reproduce the saved E3 and E8.0 predictions.
- **1 — Attention on the tumour:** Grad-CAM at the last convolutional block (`layer4`), upsampled to 96³, ReLU, normalised to
  sum 1, averaged over the 5 models; target = predicted class. Share of the map inside each phase's tumour mask, and the
  enrichment over chance (the tumour's share of the crop).
- **2 — Phase contribution:** per-phase share of |gradient × input| (each phase is its own input channel).
- **3 — Cross-phase correspondence** is not computable from Grad-CAM (phases are merged in the first layer); stated as a
  limitation and quantified with E47 (tumour overlap between phase crops).
- **Examples, chosen by rule (not by eye):** Figure 8 — per class, the correctly classified held-out patient closest to the
  class median predicted probability, plus one misclassified case; MCT-LTDiag HCC and ICC chosen the same way. Figure 2 —
  per class, the patient nearest the class median portal-venous tumour size (internal and MCT-LTDiag).
- **Claims:** descriptive only; no claim about what the model "reasons".

**Runtime:** GPU (L4 or A100), about 30–45 minutes. **Safety:** inputs are only read; outputs go to a new folder
`liver_project/revision_2026-10/F2_<time>/`. The last cell saves everything to Drive and shuts the runtime down
(set `SHUTDOWN_WHEN_DONE = False` there to keep it).

In [ ]:
# Cell 1 — Setup: installs, paths, input checks, output folder
import os, sys, json, time, hashlib, platform, datetime, warnings, subprocess
from pathlib import Path
warnings.filterwarnings('ignore')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai'], check=True)
else:
    ROOT = Path(os.environ['F2_TEST_ROOT'])
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from monai.networks.nets import resnet18
from sklearn.metrics import roc_auc_score

PROJECT_DIR = ROOT / 'liver_project'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
E6_RUN = REVISION_DIR / 'E6_20261017T114646Z'
E8A_RUN = REVISION_DIR / 'E8a_20261017T162800Z'
E4D_RUN = REVISION_DIR / 'E4d_20261019T030857Z'
MCT_DIR = PROJECT_DIR / 'external_validation' / 'MCT-LTDiag'
SEED = 42
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']
CLASSES = ['HCC', 'ICC', 'CHCC']
P3 = ['p_HCC', 'p_ICC', 'p_CHCC']
INPUTS = {
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
    'e8a_e80_predictions': (E8A_RUN / 'E8_0_E6_203_predictions.csv', '6d4195fb75e291c576f91ba93ea944e512425c5f6a0e8498b335fcddfff34daf'),
    'mct_selected': (MCT_DIR / 'metadata' / 'selected_HCC_ICC_patients.csv', 'fdb8d8fa74d8211bbfe45ecb4820415e52a5183ce5d51bcfa4376cfa9ed11e20'),
    'mct_facts': (E4D_RUN / 'E4d_mct_facts.csv', None),
}
MODEL_FILES = {f'd_{f}': E3_DIR / 'D_phase_dropout' / f'fold_{f}.pt' for f in range(5)}
def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()
problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}'); continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
problems += [f'missing: {p}' for p in MODEL_FILES.values() if not p.exists()]
tar_path = PROJECT_DIR / 'processed.tar'
if not tar_path.exists() or (IN_COLAB and tar_path.stat().st_size != 402616320):
    problems.append(f'missing or changed size: {tar_path}')
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import monai
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__, 'numpy': np.__version__,
            'device': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu'}
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = REVISION_DIR / f'F2_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=False)
print('Inputs and models checked:', len(INPUTS) + len(MODEL_FILES), '| versions:', VERSIONS, '| output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Data: internal crops (Week 2 processed), MCT-LTDiag crops (E6 + the 2 restored by E8.0; E8b's loading), labels, sizes
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(tar_path), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['F2_TEST_PROCESSED'])
split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
LABEL = internal.set_index('patient_id')['label'].astype(int).to_dict()
CT, TM, SIZE = {}, {}, {}
for pid in dev_ids + test_ids:
    with np.load(PROCESSED_DIR / f'{pid}.npz', allow_pickle=False) as z:
        CT[pid], TM[pid], SIZE[pid] = z['ct'].astype(np.uint8), z['tumor_mask'] > 0, float(z['tumor_vol_portal'])
SEL = pd.read_csv(INPUTS['mct_selected'][0], dtype=str)
MCT_IDS = SEL['patient_id'].tolist()
MCT_LABEL = dict(zip(MCT_IDS, SEL['type'].map({'HCC': 0, 'ICC': 1})))
MCT_CT, MCT_TM = {}, {}
for pid in MCT_IDS:
    path = E6_RUN / 'crops_primary' / f'{pid}.npz'
    if not path.exists():
        path = E8A_RUN / 'crops_primary_restored2' / f'{pid}.npz'
    with np.load(path, allow_pickle=False) as z:
        MCT_CT[pid] = z['ct'].astype(np.uint8)
        tm = z['tumor_mask'] > 0
        MCT_TM[pid] = np.broadcast_to(tm, (4,) + tm.shape) if tm.ndim == 3 else tm      # one portal-venous mask, phases aligned by E6
if len(MCT_CT) != 203 and IN_COLAB:
    raise RuntimeError(f'STOP — {len(MCT_CT)} MCT-LTDiag crops found, expected 203.')
facts = pd.read_csv(INPUTS['mct_facts'][0], dtype={'patient_id': str}).set_index('patient_id')
MCT_SIZE = facts['tumor_volume_ml'].reindex(MCT_IDS).astype(float).to_dict()
print(f'Internal crops {len(CT)} (held-out {len(test_ids)}), MCT-LTDiag crops {len(MCT_CT)}')

In [ ]:
# Cell 3 — Frozen main CNN; gate: the ensemble reproduces the saved E3 (held-out) and E8.0 (MCT-LTDiag) predictions
def zscore4(x4):                              # Week 4 / E3, unchanged
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def make_model():
    return resnet18(spatial_dims=3, n_input_channels=4, num_classes=3, feed_forward=True)

def load_state(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model = make_model().to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return model

MODELS = [load_state(MODEL_FILES[f'd_{f}']) for f in range(5)]

@torch.no_grad()
def ensemble_probs(ct4):
    x = torch.from_numpy(zscore4(ct4.astype(np.float32))).unsqueeze(0).to(DEVICE)
    return np.mean([torch.softmax(m(x).float(), dim=1).cpu().numpy()[0] for m in MODELS], axis=0)

PROB_INT = pd.DataFrame({p: ensemble_probs(CT[p]) for p in test_ids}, index=P3).T
PROB_MCT = pd.DataFrame({p: ensemble_probs(MCT_CT[p]) for p in MCT_IDS}, index=P3).T
d = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
d = d.query("pattern == 'complete' and split == 'test'").set_index('patient_id').loc[test_ids, P3]
e80 = pd.read_csv(INPUTS['e8a_e80_predictions'][0], dtype={'patient_id': str})
e80 = e80[(e80['variant'] == 'primary') & (e80['model'] == 'phase_dropout_CNN')].set_index('patient_id').loc[MCT_IDS, P3]
GATE = {'internal_vs_E3': float(np.abs(PROB_INT.to_numpy() - d.to_numpy()).max()),
        'MCT_vs_E8_0': float(np.abs(PROB_MCT.loc[MCT_IDS].to_numpy() - e80.to_numpy()).max())}
print('Gate — ensemble vs saved predictions (max difference, must be < 0.02):', GATE)
if IN_COLAB and max(GATE.values()) > 0.02:
    raise RuntimeError('STOP — the frozen CNN does not reproduce its saved predictions. Send this output.')

In [ ]:
# Cell 4 — Grad-CAM (layer4) and per-phase |gradient x input| for the held-out 56 and MCT-LTDiag 203
def maps(ct4, target):
    """Ensemble Grad-CAM (normalised to sum 1, averaged over the 5 models) and per-phase attribution shares."""
    x0 = torch.from_numpy(zscore4(ct4.astype(np.float32))).unsqueeze(0).to(DEVICE)
    cams, shares = [], []
    for m in MODELS:
        store = {}
        hook = m.layer4.register_forward_hook(lambda module, inp, out: store.__setitem__('a', out))
        x = x0.clone().requires_grad_(True)
        score = m(x)[0, target]
        hook.remove()
        grad_a, grad_x = torch.autograd.grad(score, [store['a'], x])
        weights = grad_a.mean(dim=(2, 3, 4), keepdim=True)
        cam = torch.relu((weights * store['a']).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=tuple(x.shape[2:]), mode='trilinear', align_corners=False)[0, 0]
        total = cam.sum()
        cams.append((cam / total).detach().cpu().numpy() if total > 0 else np.zeros(tuple(x.shape[2:]), np.float32))
        att = (grad_x * x).abs()[0].sum(dim=(1, 2, 3))
        shares.append((att / att.sum()).detach().cpu().numpy() if att.sum() > 0 else np.full(4, np.nan))
    return np.mean(cams, axis=0), np.nanmean(shares, axis=0)

rows, share_rows, CAMS = [], [], {}
t0 = time.time()
for cohort, ids, cts, tms, labels, probs in [('internal_held_out', test_ids, CT, TM, LABEL, PROB_INT), ('MCT_LTDiag', MCT_IDS, MCT_CT, MCT_TM, MCT_LABEL, PROB_MCT)]:
    for n, pid in enumerate(ids, start=1):
        p = probs.loc[pid].to_numpy(); pred = int(p.argmax()); true = int(labels[pid])
        black = bool(cts[pid].max() == 0)                       # e.g. empty liver mask (E10a): attention undefined
        cam, share = maps(cts[pid], pred)
        CAMS[(cohort, pid, 'pred')] = cam
        if pred != true:
            CAMS[(cohort, pid, 'true')] = maps(cts[pid], true)[0]
        for i, ph in enumerate(PHASE_ORDER):
            tm = tms[pid][i]
            chance = float(tm.mean())
            inside = float(cam[tm].sum()) if chance > 0 else np.nan
            rows.append({'cohort': cohort, 'patient_id': pid, 'class': CLASSES[true], 'predicted': CLASSES[pred], 'correct': pred == true,
                         'black_crop': black, 'phase': ph, 'attention_in_tumour': inside, 'tumour_share_of_crop': chance,
                         'enrichment': inside / chance if chance > 0 and not black else np.nan})
        share_rows.append({'cohort': cohort, 'patient_id': pid, 'class': CLASSES[true], 'predicted': CLASSES[pred], 'correct': pred == true,
                           'black_crop': black, **{f'share_{ph}': float(share[i]) for i, ph in enumerate(PHASE_ORDER)}})
        if n % 25 == 0:
            print(f'  {cohort}: {n}/{len(ids)} ({(time.time() - t0) / 60:.1f} min)', flush=True)
ATT = pd.DataFrame(rows); SHARE = pd.DataFrame(share_rows)
ATT.to_csv(OUT_DIR / 'F2_attention_per_patient_phase.csv', index=False)
SHARE.to_csv(OUT_DIR / 'F2_phase_share_per_patient.csv', index=False)
ok = ATT[~ATT['black_crop']]
SUM_ATT = ok.groupby(['cohort', 'class', 'phase'])['enrichment'].describe(percentiles=[.25, .5, .75])[['count', '25%', '50%', '75%']].rename(
    columns={'50%': 'median_enrichment', '25%': 'q1', '75%': 'q3'})
SUM_ATT['median_attention_in_tumour'] = ok.groupby(['cohort', 'class', 'phase'])['attention_in_tumour'].median()
SUM_SHARE = SHARE[~SHARE['black_crop']].groupby(['cohort', 'class'])[[f'share_{ph}' for ph in PHASE_ORDER]].median()
SUM_ATT.to_csv(OUT_DIR / 'F2_attention_summary.csv'); SUM_SHARE.to_csv(OUT_DIR / 'F2_phase_share_summary.csv')
print('Black crops excluded from the summaries:', sorted(set(ATT.loc[ATT['black_crop'], 'patient_id'])))
print('Attention on the tumour (enrichment over chance; median, IQR):\n', SUM_ATT.round(3).to_string())
print('Phase contribution (median share of |gradient x input|):\n', SUM_SHARE.round(3).to_string())

In [ ]:
# Cell 5 — Examples chosen by rule; save slices for Figures 2 and 8 (drawn locally by figures_revision.py)
def closest_to_median(cands, values):
    if not cands:
        return None
    v = pd.Series({c: values[c] for c in cands})
    return (v - v.median()).abs().idxmin()
EX, META = {}, []
def add_example(tag, cohort, pid, ct4, tm4, cam_pred, cam_true, probs, true, pred, note=''):
    tz = tm4[2] if tm4[2].any() else tm4.any(axis=0)
    z = int(round(np.argwhere(tz)[:, 2].mean())) if tz.any() else ct4.shape[3] // 2
    EX[f'{tag}_ct'] = ct4[:, :, :, z]
    EX[f'{tag}_tm'] = np.stack([t[:, :, z] for t in tm4]).astype(np.uint8)
    if cam_pred is not None:
        EX[f'{tag}_cam_pred'] = cam_pred[:, :, z].astype(np.float32)
    if cam_true is not None:
        EX[f'{tag}_cam_true'] = cam_true[:, :, z].astype(np.float32)
    META.append({'tag': tag, 'cohort': cohort, 'patient_id': pid, 'class': CLASSES[true], 'predicted': CLASSES[pred] if pred is not None else '',
                 'probabilities': [round(float(x), 4) for x in probs] if probs is not None else [], 'slice': z, 'note': note})
# Figure 8: held-out — per class, the correct case closest to the class median predicted probability; one misclassified case
for k, c in enumerate(CLASSES):
    members = [p for p in test_ids if LABEL[p] == k and CT[p].max() > 0]
    correct = [p for p in members if PROB_INT.loc[p].to_numpy().argmax() == k]
    pool, note = (correct, 'correct case closest to the median') if correct else (members, 'no correctly classified case; closest to the class median')
    pid = closest_to_median(pool, {p: PROB_INT.loc[p].iloc[k] for p in pool})
    if pid:
        pred = int(PROB_INT.loc[pid].to_numpy().argmax())
        add_example(f'fig8_internal_{c}', 'internal_held_out', pid, CT[pid], TM[pid], CAMS[('internal_held_out', pid, 'pred')],
                    CAMS.get(('internal_held_out', pid, 'true')), PROB_INT.loc[pid], k, pred, note)
wrong = [p for p in test_ids if PROB_INT.loc[p].to_numpy().argmax() != LABEL[p] and CT[p].max() > 0]
pid = closest_to_median(wrong, {p: PROB_INT.loc[p].max() for p in wrong})
if pid:
    add_example('fig8_internal_misclassified', 'internal_held_out', pid, CT[pid], TM[pid], CAMS[('internal_held_out', pid, 'pred')],
                CAMS[('internal_held_out', pid, 'true')], PROB_INT.loc[pid], LABEL[pid], int(PROB_INT.loc[pid].to_numpy().argmax()))
for k, c in enumerate(['HCC', 'ICC']):
    members = [p for p in MCT_IDS if MCT_LABEL[p] == k]
    correct = [p for p in members if PROB_MCT.loc[p].to_numpy().argmax() == k]
    pool, note = (correct, 'correct case closest to the median') if correct else (members, 'no correctly classified case; closest to the class median')
    pid = closest_to_median(pool, {p: PROB_MCT.loc[p].iloc[k] for p in pool})
    pred = int(PROB_MCT.loc[pid].to_numpy().argmax())
    add_example(f'fig8_mct_{c}', 'MCT_LTDiag', pid, MCT_CT[pid], MCT_TM[pid], CAMS[('MCT_LTDiag', pid, 'pred')],
                CAMS.get(('MCT_LTDiag', pid, 'true')), PROB_MCT.loc[pid], k, pred, note)
# Figure 2: per class, the patient nearest the class median portal-venous tumour size (all internal; MCT-LTDiag by volume)
for k, c in enumerate(CLASSES):
    members = [p for p in dev_ids + test_ids if LABEL[p] == k and CT[p].max() > 0]
    pid = closest_to_median(members, SIZE)
    add_example(f'fig2_internal_{c}', 'internal', pid, CT[pid], TM[pid], None, None, None, k, None, 'nearest the class median tumour size')
for k, c in enumerate(['HCC', 'ICC']):
    members = [p for p in MCT_IDS if MCT_LABEL[p] == k and np.isfinite(MCT_SIZE.get(p, np.nan))]
    pid = closest_to_median(members, MCT_SIZE)
    add_example(f'fig2_mct_{c}', 'MCT_LTDiag', pid, MCT_CT[pid], MCT_TM[pid], None, None, None, k, None, 'nearest the class median tumour volume')
np.savez_compressed(OUT_DIR / 'F2_examples.npz', **EX)
(OUT_DIR / 'F2_examples.json').write_text(json.dumps(META, indent=1))
print('Examples chosen by rule:'); print(pd.DataFrame(META).drop(columns=['probabilities']).to_string(index=False))

In [ ]:
# Cell 6 — Summary, manifest, results to copy
summary = {'run_id': RUN_ID, 'notebook': 'F2_gradcam_attribution_examples.ipynb', 'versions': VERSIONS, 'input_sha256': input_hashes,
           'gates': GATE, 'n_examples': len(META), 'protocol': '00 log section 25 (approved 19 Oct 2026); frozen main CNN; descriptive only'}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)} for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest).to_csv(OUT_DIR / 'output_manifest.csv', index=False)
print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run F2_{RUN_ID} | gates: {GATE}')
print('\n--- attention on the tumour (enrichment over chance) ---'); print(SUM_ATT.round(3).to_string())
print('\n--- phase contribution (median share) ---'); print(SUM_SHARE.round(3).to_string())
print('\n--- examples ---'); print(pd.DataFrame(META).drop(columns=['probabilities']).to_string(index=False))
print('==============================================================')

In [ ]:
# Cell 7 — Finish: make sure every result file has reached Google Drive, then shut the runtime down (stops compute units)
SHUTDOWN_WHEN_DONE = True     # set to False to keep the runtime after the run
if IN_COLAB and SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print('All files are saved in Drive. The runtime shuts down in 30 seconds; the outputs above stay visible on this page.')
    time.sleep(30)
    runtime.unassign()